# 01 · Entrada: contrato de dados, validação e divisão por safra

**Etapa 1 do pipeline.** Recebe a base em Spark e entrega a base pronta para modelagem, com cada contrato marcado como `treino`, `teste`, `oot` ou `hist`.

| passo | o que faz | por quê |
|---|---|---|
| 1 | identifica as variáveis explicativas | tudo que não é coluna de controle é candidato a variável |
| 2 | calcula colunas derivadas (idade, evento, maturidade) | a PD lifetime e as verificações precisam delas |
| 3 | confere o contrato de dados | erro de dado aqui contamina todas as etapas seguintes |
| 4 | exclui safras imaturas | alvo incompleto parece risco baixo sem ser |
| 5 | divide o DES em treino e teste pela ordem das safras | o teste fica **depois** do treino no tempo, como o uso real |
| 6 | resume a base por partição e por safra | primeira leitura da base: volumes e taxas |

**Falha × aviso.** Uma verificação do tipo *falha* interrompe o pipeline: a base não pode seguir assim. Uma do tipo *aviso* só registra e conta os casos.

## Contrato de dados (só colunas essenciais)

**PD**: uma linha por contrato.

| coluna | tipo | regra |
|---|---|---|
| `id_contrato` | texto | única |
| `safra` | data, 1º dia do mês | mês de concessão |
| `amostra` | texto | `DES`, `OOT` ou `HIST` (safras anteriores ao DES, usadas só no teste da PD lifetime) |
| `alvo_12m` | inteiro 0/1 | 1 se o primeiro default ocorreu até 12 meses após a concessão; vazio em safra imatura |
| `data_primeiro_default` | data, 1º dia do mês | primeiro mês em default (ex.: 90 dias de atraso); vazia se nunca |
| `data_fim_observacao` | data, 1º dia do mês | igual a `data_primeiro_default` se houve default; senão, o menor entre a data de corte, a quitação e outra saída |
| variáveis explicativas | número ou texto | medidas na data da concessão; as mesmas em DES, OOT e HIST |

**LGD**: uma linha por contrato **inadimplente com cobrança encerrada**.

| coluna | tipo | regra |
|---|---|---|
| `id_contrato` | texto | única |
| `safra_default` | data, 1º dia do mês | mês do default |
| `amostra` | texto | `DES` ou `OOT` |
| `lgd_realizada` | decimal em [0, 1] | perda final da cobrança / exposição no default |
| variáveis explicativas | número ou texto | medidas na data do default |

**Convenção de idade.** A idade é o número de meses-calendário desde a safra: o mês seguinte à concessão é a idade 1. Um contrato concedido em mar/23 que entrou em default em out/23 inadimpliu na idade 7.

In [ ]:
%run ./00_config

In [ ]:
from datetime import date

import numpy as np
import pandas as pd
from pyspark.sql import functions as F

# Colunas criadas por esta etapa. Ficam fora da lista de variáveis explicativas.
COLUNAS_DERIVADAS = [
    "meses_ate_corte", "safra_madura", "evento_default", "meses_ate_default",
    "idade_fim_observacao", "alvo_esperado_12m", "particao",
]

# Ordem de exibição das partições.
ORDEM_PARTICOES = ["treino", "teste", "oot", "hist"]

## 1. Variáveis explicativas

Se a configuração trouxer a lista de variáveis (`cfg["variaveis"]`), ela é usada. Senão, toda coluna que não é de controle (id, safra, amostra, alvo, datas) nem derivada é tratada como variável explicativa candidata. A seleção de verdade acontece nas etapas 2 e 3.

In [ ]:
def colunas_de_controle(cfg):
    """Colunas do contrato de dados que NÃO são variáveis explicativas."""
    return [nome for nome in cfg["colunas"].values() if nome]


def lista_variaveis_explicativas(sdf, cfg):
    """
    Lista as variáveis explicativas candidatas.

    Conceito de negócio:
        Variável explicativa é uma característica do contrato ou do cliente
        conhecida no momento da decisão: na concessão (PD) ou no default (LGD).
        Uma coluna preenchida depois disso (por exemplo, um atraso posterior)
        carrega informação do futuro e gera VAZAMENTO: o modelo parece ótimo no
        desenvolvimento e falha em produção. A etapa 2 marca suspeitas de
        vazamento, mas a garantia de que a base só tem informação do passado é
        de quem monta a base.
    """
    if cfg["variaveis"] is not None:
        return list(cfg["variaveis"])
    excluir = set(colunas_de_controle(cfg)) | set(COLUNAS_DERIVADAS)
    return [coluna for coluna in sdf.columns if coluna not in excluir]

## 2. Colunas derivadas: idade, evento e maturidade

A partir das datas do contrato, calcula:

| coluna | tarefa | significado |
|---|---|---|
| `meses_ate_corte` | PD e LGD | meses entre a safra e a data de corte: quanto histórico a safra tem |
| `safra_madura` | PD e LGD | PD: ≥ 12 meses de desempenho. LGD: ≥ 24 meses de cobrança |
| `evento_default` | PD | 1 se o contrato entrou em default em algum momento até o corte |
| `meses_ate_default` | PD | idade em que entrou em default (vazia se nunca) |
| `idade_fim_observacao` | PD | idade no último mês acompanhado: a "duração" da análise de sobrevivência |
| `alvo_esperado_12m` | PD | o alvo de 12 meses recalculado pelas datas, para conferir o `alvo_12m` informado |

As três últimas formam o **formato de sobrevivência** que a PD lifetime (etapa 8) usa: para cada contrato, até que idade foi acompanhado e se terminou em default ou foi censurado (saiu da observação sem inadimplir).

In [ ]:
def meses_entre(inicio, fim):
    """
    Número de meses-calendário de `inicio` até `fim` (colunas Spark ou nomes).

    Exemplo: de 2023-03-01 a 2023-10-01 → 7. Como todas as datas do pipeline
    são o 1º dia do mês, a conta é exata e não depende do número de dias.
    """
    inicio = F.col(inicio) if isinstance(inicio, str) else inicio
    fim = F.col(fim) if isinstance(fim, str) else fim
    return (F.year(fim) - F.year(inicio)) * 12 + (F.month(fim) - F.month(inicio))


def adiciona_colunas_derivadas(sdf, cfg):
    """
    Acrescenta as colunas de idade, evento e maturidade.

    Conceito de negócio — maturidade:
        PD: o alvo de 12 meses só é válido se a safra já teve 12 meses de
        desempenho até a data de corte. Uma safra de 5 meses tem alvo 0 para
        muitos contratos que ainda vão inadimplir: a taxa sai baixa por falta de
        tempo, e não por bom risco.
        LGD: a LGD realizada só é confiável se a cobrança teve tempo de terminar
        (janela de 24 meses). Nas safras de default recentes, as cobranças já
        encerradas são as curas rápidas ou as baixas rápidas, e a média engana.

    Conceito técnico — alvo esperado:
        alvo_esperado_12m = 1 quando o default aconteceu até a idade 12
        (inclusive). Serve só para CONFERIR o alvo informado; o pipeline segue
        com o alvo da base e registra quantos contratos divergem.
    """
    colunas = cfg["colunas"]
    data_corte = F.to_date(F.lit(cfg["data_corte"]))
    safra = F.col(colunas["safra"])
    sdf = sdf.withColumn("meses_ate_corte", meses_entre(safra, data_corte))

    if cfg["tarefa"] == "pd":
        janela = cfg["entrada"]["janela_desempenho_meses"]
        data_default = F.col(colunas["data_primeiro_default"])
        sdf = (sdf
               .withColumn("safra_madura", F.col("meses_ate_corte") >= janela)
               .withColumn("evento_default", F.when(data_default.isNotNull(), 1).otherwise(0))
               .withColumn("meses_ate_default", meses_entre(safra, data_default))
               .withColumn("idade_fim_observacao",
                           meses_entre(safra, F.col(colunas["data_fim_observacao"])))
               .withColumn("alvo_esperado_12m",
                           F.when(~F.col("safra_madura"), F.lit(None).cast("int"))
                            .when((F.col("evento_default") == 1)
                                  & (F.col("meses_ate_default") <= janela), 1)
                            .otherwise(0)))
    else:
        janela = cfg["entrada"]["janela_cobranca_meses"]
        sdf = sdf.withColumn("safra_madura", F.col("meses_ate_corte") >= janela)
    return sdf

## 3. Verificações do contrato de dados

Todas as regras de linha são contadas numa **única passada** pela base (uma agregação Spark com uma soma condicional por regra). A ordem das safras e as variáveis vazias exigem uma agregação por amostra cada.

| grupo | verificações |
|---|---|
| estrutura | colunas obrigatórias presentes; tipos (datas como `date`, alvo numérico) |
| identificação | id vazio; id repetido; amostra fora dos rótulos aceitos |
| datas | safra vazia, fora do 1º dia do mês ou posterior ao corte; default antes da safra; fim de observação vazio, antes da safra ou depois do corte; contrato em default acompanhado depois do default |
| alvo | fora de {0, 1} (PD) ou de [0, 1] (LGD); vazio em safra madura; preenchido em safra imatura; incoerente com as datas |
| amostras | DES e OOT presentes; ordem HIST < DES < OOT |
| variáveis | totalmente vazias em DES ou OOT (falha) ou em HIST (aviso) |

In [ ]:
TIPOS_DATA = ("date", "timestamp")


def _eh_numerico(tipo_spark):
    """True para os tipos numéricos do Spark (int, bigint, double, decimal(p,s)...)."""
    return (tipo_spark in ("tinyint", "smallint", "int", "bigint", "float", "double")
            or tipo_spark.startswith("decimal"))


def _linha_verificacao(verificacao, tipo, n_violacoes, detalhe=""):
    """Uma linha do relatório: status 'ok' se não houve violação; senão o próprio tipo."""
    n = int(n_violacoes or 0)
    status = "ok" if n == 0 else tipo
    return {"verificacao": verificacao, "tipo": tipo, "status": status,
            "n_violacoes": n, "detalhe": detalhe}


def valida_estrutura(sdf, cfg, variaveis):
    """
    Primeira barreira: colunas presentes e tipos corretos. Se falhar aqui, as
    demais verificações nem rodam, porque dependem das datas e do alvo.
    """
    colunas = cfg["colunas"]
    linhas = []
    faltando = [nome for nome in colunas_de_controle(cfg) if nome not in sdf.columns]
    faltando += [v for v in variaveis if v not in sdf.columns]
    linhas.append(_linha_verificacao("colunas obrigatórias e variáveis da configuração presentes",
                                     "falha", len(faltando), ", ".join(faltando)))
    linhas.append(_linha_verificacao("há variáveis explicativas", "falha",
                                     0 if variaveis else 1,
                                     "" if variaveis else "nenhuma coluna além das de controle"))
    if faltando or not variaveis:
        return linhas

    tipos = dict(sdf.dtypes)
    colunas_data = [colunas["safra"]]
    if cfg["tarefa"] == "pd":
        colunas_data += [colunas["data_primeiro_default"], colunas["data_fim_observacao"]]
    tipo_errado = [f"{nome} ({tipos[nome]})" for nome in colunas_data if tipos[nome] not in TIPOS_DATA]
    if not _eh_numerico(tipos[colunas["alvo"]]):
        tipo_errado.append(f"{colunas['alvo']} ({tipos[colunas['alvo']]})")
    linhas.append(_linha_verificacao("tipos das colunas (datas como date, alvo numérico)",
                                     "falha", len(tipo_errado), ", ".join(tipo_errado)))
    return linhas


def _regras_de_linha(cfg):
    """
    Regras verificadas linha a linha: (nome, tipo, condição Spark, detalhe).
    A condição é VERDADEIRA quando a linha VIOLA a regra.
    """
    colunas, rotulos = cfg["colunas"], cfg["rotulos_amostra"]
    amostra = F.col(colunas["amostra"])
    safra = F.col(colunas["safra"])
    alvo = F.col(colunas["alvo"])
    data_corte = F.to_date(F.lit(cfg["data_corte"]))

    if cfg["tarefa"] == "pd":
        aceitos = [rotulos["des"], rotulos["oot"], rotulos["hist"]]
    else:
        aceitos = [rotulos["des"], rotulos["oot"]]

    regras = [
        ("id_contrato vazio", "falha", F.col(colunas["id"]).isNull(), ""),
        ("amostra fora dos rótulos aceitos", "falha",
         amostra.isNull() | ~amostra.isin(*aceitos), f"aceitos: {aceitos}"),
        ("safra vazia", "falha", safra.isNull(), ""),
        ("safra não é o 1º dia do mês", "falha", F.dayofmonth(safra) != 1, ""),
        ("safra posterior à data de corte", "falha", safra > data_corte, ""),
    ]

    if cfg["tarefa"] == "pd":
        janela = cfg["entrada"]["janela_desempenho_meses"]
        data_default = F.col(colunas["data_primeiro_default"])
        data_fim = F.col(colunas["data_fim_observacao"])
        modelagem = amostra.isin(rotulos["des"], rotulos["oot"])
        madura = F.col("safra_madura")
        regras += [
            ("data_primeiro_default não é o 1º dia do mês", "falha",
             data_default.isNotNull() & (F.dayofmonth(data_default) != 1), ""),
            ("data_primeiro_default anterior à safra", "falha", data_default < safra, ""),
            ("data_primeiro_default posterior à data de corte", "falha", data_default > data_corte, ""),
            ("data_fim_observacao vazia", "falha", data_fim.isNull(), ""),
            ("data_fim_observacao não é o 1º dia do mês", "falha", F.dayofmonth(data_fim) != 1, ""),
            ("data_fim_observacao anterior à safra", "falha", data_fim < safra, ""),
            ("data_fim_observacao posterior à data de corte", "falha", data_fim > data_corte, ""),
            # Default é absorvente: depois de entrar em default, o contrato sai da
            # análise. Acompanhá-lo além disso contaria o mesmo default de novo.
            ("contrato em default acompanhado depois do default", "falha",
             data_default.isNotNull() & (data_fim != data_default),
             "data_fim_observacao deve ser igual a data_primeiro_default"),
            ("alvo_12m fora de {0, 1}", "falha", alvo.isNotNull() & ~alvo.isin(0, 1), ""),
            ("alvo_12m vazio em safra madura de DES/OOT", "falha",
             modelagem & madura & alvo.isNull(), "contratos sem alvo não podem ser modelados"),
            ("contratos de DES/OOT em safra imatura (serão excluídos)", "aviso",
             modelagem & ~madura, f"menos de {janela} meses até a data de corte"),
            ("alvo_12m preenchido em safra imatura", "aviso",
             ~madura & alvo.isNotNull(), "o alvo pode estar incompleto"),
            ("alvo_12m incoerente com as datas", "aviso",
             alvo.isNotNull() & F.col("alvo_esperado_12m").isNotNull()
             & (alvo != F.col("alvo_esperado_12m")),
             "o pipeline segue com o alvo informado; conferir a extração"),
        ]
    else:
        janela = cfg["entrada"]["janela_cobranca_meses"]
        regras += [
            ("lgd_realizada vazia", "falha", alvo.isNull(), ""),
            ("lgd_realizada fora de [0, 1]", "falha", (alvo < 0) | (alvo > 1), ""),
            ("safras de default imaturas (serão excluídas)", "aviso",
             ~F.col("safra_madura"), f"menos de {janela} meses de cobrança até a data de corte"),
        ]
    return regras


def valida_contrato_entrada(sdf, cfg, variaveis):
    """
    Confere a base contra o contrato de dados e devolve o relatório (pandas)
    com uma linha por verificação: verificacao, tipo, status, n_violacoes, detalhe.

    Conceito técnico:
        As regras de linha viram somas condicionais numa única agregação Spark
        (uma passada pela base, qualquer que seja o número de regras). Uma
        condição que dá NULO (por exemplo, comparar uma data vazia) não conta
        como violação; os vazios têm regras próprias.
    """
    linhas = valida_estrutura(sdf, cfg, variaveis)
    if any(linha["status"] == "falha" for linha in linhas):
        return pd.DataFrame(linhas)

    colunas, rotulos = cfg["colunas"], cfg["rotulos_amostra"]
    base = adiciona_colunas_derivadas(sdf, cfg)

    # ---- regras de linha: uma passada ----------------------------------------
    regras = _regras_de_linha(cfg)
    expressoes = [F.sum(F.when(condicao, 1).otherwise(0)).alias(f"regra_{i}")
                  for i, (_, _, condicao, _) in enumerate(regras)]
    expressoes += [F.count(F.lit(1)).alias("n_linhas"),
                   F.countDistinct(colunas["id"]).alias("n_ids_distintos")]
    contagem = base.agg(*expressoes).collect()[0]

    for i, (nome, tipo, _, detalhe) in enumerate(regras):
        linhas.append(_linha_verificacao(nome, tipo, contagem[f"regra_{i}"], detalhe))

    # id repetido: linhas com id preenchido menos ids distintos.
    n_id_vazio = contagem["regra_0"] or 0
    repetidos = contagem["n_linhas"] - n_id_vazio - contagem["n_ids_distintos"]
    linhas.append(_linha_verificacao("id_contrato repetido", "falha", repetidos,
                                     "linhas a mais além da primeira de cada id"))

    # ---- amostras presentes e ordem das safras --------------------------------
    limites = (base.groupBy(colunas["amostra"])
                   .agg(F.min(colunas["safra"]).alias("primeira"),
                        F.max(colunas["safra"]).alias("ultima"))
                   .toPandas().set_index(colunas["amostra"]))
    ausentes = [r for r in (rotulos["des"], rotulos["oot"]) if r not in limites.index]
    linhas.append(_linha_verificacao("amostras DES e OOT presentes", "falha",
                                     len(ausentes), ", ".join(ausentes)))
    fora_de_ordem = []
    if not ausentes:
        if limites.loc[rotulos["des"], "ultima"] >= limites.loc[rotulos["oot"], "primeira"]:
            fora_de_ordem.append("a última safra do DES não é anterior à primeira do OOT")
        if (cfg["tarefa"] == "pd" and rotulos["hist"] in limites.index
                and limites.loc[rotulos["hist"], "ultima"] >= limites.loc[rotulos["des"], "primeira"]):
            fora_de_ordem.append("a última safra do HIST não é anterior à primeira do DES")
    rotulo_ordem = "ordem das safras (HIST < DES < OOT)" if cfg["tarefa"] == "pd" \
        else "ordem das safras (DES < OOT)"
    linhas.append(_linha_verificacao(rotulo_ordem, "falha",
                                     len(fora_de_ordem), "; ".join(fora_de_ordem)))
    if cfg["tarefa"] == "pd":
        linhas.append(_linha_verificacao(
            "amostra HIST presente", "aviso", 0 if rotulos["hist"] in limites.index else 1,
            "sem HIST, os horizontes longos da PD lifetime são testados só até onde DES e OOT alcançam"))

    # ---- variáveis totalmente vazias por amostra ------------------------------
    # F.count(coluna) conta só os valores preenchidos: 0 = variável toda vazia.
    preenchidos = (base.groupBy(colunas["amostra"])
                       .agg(*[F.count(F.col(v)).alias(v) for v in variaveis])
                       .toPandas().set_index(colunas["amostra"]))
    vazias_modelagem = [f"{v} ({a})" for a in (rotulos["des"], rotulos["oot"])
                        if a in preenchidos.index for v in variaveis if preenchidos.loc[a, v] == 0]
    linhas.append(_linha_verificacao("variáveis totalmente vazias em DES ou OOT", "falha",
                                     len(vazias_modelagem), ", ".join(vazias_modelagem[:20])))
    if cfg["tarefa"] == "pd" and rotulos["hist"] in preenchidos.index:
        vazias_hist = [v for v in variaveis if preenchidos.loc[rotulos["hist"], v] == 0]
        detalhe_hist = ("contratos HIST sem essas variáveis não podem ser escorados e ficam fora "
                        "do backtest por rating: " + ", ".join(vazias_hist[:20])) if vazias_hist else ""
        linhas.append(_linha_verificacao("variáveis totalmente vazias em HIST", "aviso",
                                         len(vazias_hist), detalhe_hist))

    return pd.DataFrame(linhas)


def interrompe_se_falha(relatorio):
    """Levanta erro listando as verificações com status 'falha'."""
    falhas = relatorio[relatorio["status"] == "falha"]
    if len(falhas):
        texto = "\n".join(f"  - {linha.verificacao}: {linha.n_violacoes} "
                          f"{('(' + linha.detalhe + ')') if linha.detalhe else ''}"
                          for linha in falhas.itertuples())
        raise ValueError("A base de entrada não cumpre o contrato de dados:\n" + texto)

## 4. Exclusão de safras imaturas

- **PD:** saem os contratos de DES e OOT cuja safra tem menos de 12 meses até a data de corte. As safras HIST ficam inteiras: a PD lifetime usa o histórico que cada safra tem, e a censura cuida das que têm pouco tempo.
- **LGD:** saem as safras de default com menos de 24 meses até o corte.

O resumo do que saiu, por amostra e safra, é exibido e gravado.

In [ ]:
def separa_safras_imaturas(base, cfg):
    """
    Remove os contratos de safras imaturas e devolve (base_filtrada, resumo_excluidos).

    Conceito de negócio:
        Manter uma safra imatura no DES ensina ao modelo que aquele perfil tem
        risco menor do que tem. No OOT, faz o modelo parecer melhor do que é.
    """
    colunas, rotulos = cfg["colunas"], cfg["rotulos_amostra"]
    if cfg["tarefa"] == "pd":
        excluir = (F.col(colunas["amostra"]).isin(rotulos["des"], rotulos["oot"])
                   & ~F.col("safra_madura"))
    else:
        excluir = ~F.col("safra_madura")
    resumo = (base.filter(excluir)
                  .groupBy(colunas["amostra"], colunas["safra"])
                  .agg(F.count(F.lit(1)).alias("contratos_excluidos"))
                  .orderBy(colunas["amostra"], colunas["safra"])
                  .toPandas())
    return base.filter(~excluir), resumo

## 5. Divisão treino/teste pela ordem das safras

As safras do DES são ordenadas no tempo e cortadas **entre duas safras**, no ponto cujo volume acumulado fica mais perto de 80% dos contratos. Uma safra nunca é partida.

Exemplo: DES com 12 safras de cerca de 166.700 contratos cada.

| corte depois de | safras no treino | % do volume | distância de 80% |
|---|---|---|---|
| nov/23 | 9 | 75% | 5,0 p.p. |
| **dez/23** | **10** | **83%** | **3,3 p.p.** ← escolhido |

Resultado: treino de mar/23 a dez/23, teste de jan/24 a fev/24. Em caso de empate na distância, fica o corte com menos safras no treino.

In [ ]:
def escolhe_corte_treino_teste(contagem_por_safra, proporcao_treino):
    """
    Escolhe a última safra do treino.

    Parâmetros
        contagem_por_safra  pandas com colunas "safra" e "n" (contratos do DES).
        proporcao_treino    fração-alvo do volume no treino (0,80).

    Devolve
        (ultima_safra_treino, tabela) — a tabela traz, por safra, o volume, a
        participação acumulada e a partição atribuída.

    Conceito técnico:
        Os candidatos são os cortes depois de cada safra, exceto a última (que
        deixaria o teste vazio). O escolhido é o de participação acumulada mais
        próxima da proporção-alvo. idxmin devolve o primeiro em caso de empate.
    """
    tabela = contagem_por_safra.sort_values("safra").reset_index(drop=True)
    if len(tabela) < 2:
        raise ValueError("O DES precisa de pelo menos 2 safras para separar treino e teste no tempo.")
    tabela["participacao_acumulada"] = tabela["n"].cumsum() / tabela["n"].sum()
    candidatos = tabela.iloc[:-1]
    posicao_corte = (candidatos["participacao_acumulada"] - proporcao_treino).abs().idxmin()
    tabela["particao"] = np.where(tabela.index <= posicao_corte, "treino", "teste")
    return tabela.loc[posicao_corte, "safra"], tabela


def divide_treino_teste_por_safra(base, cfg):
    """
    Cria a coluna `particao` ∈ {treino, teste, oot, hist}.

    Conceito de negócio:
        Com o teste depois do treino no tempo, toda etapa que usa o teste
        (otimização do vencedor, calibração, busca de inversões na régua) mede o
        modelo em safras que ele não viu e que vieram depois, como acontece em
        produção. O OOT continua sendo o período mais à frente.

    Devolve
        (base_com_particao, resumo_da_divisao, tabela_por_safra)
    """
    colunas, rotulos = cfg["colunas"], cfg["rotulos_amostra"]
    contagem = (base.filter(F.col(colunas["amostra"]) == rotulos["des"])
                    .groupBy(colunas["safra"])
                    .agg(F.count(F.lit(1)).alias("n"))
                    .toPandas()
                    .rename(columns={colunas["safra"]: "safra"}))
    ultima_safra_treino, tabela = escolhe_corte_treino_teste(
        contagem, cfg["divisao"]["proporcao_treino"])

    amostra = F.col(colunas["amostra"])
    particao = (F.when((amostra == rotulos["des"])
                       & (F.col(colunas["safra"]) <= F.lit(ultima_safra_treino)), "treino")
                 .when(amostra == rotulos["des"], "teste")
                 .when(amostra == rotulos["oot"], "oot")
                 .when(amostra == rotulos["hist"], "hist"))
    base = base.withColumn("particao", particao)

    no_treino = tabela["particao"] == "treino"
    resumo = {
        "ultima_safra_treino": str(ultima_safra_treino),
        "primeira_safra_teste": str(tabela.loc[~no_treino, "safra"].min()),
        "n_safras_treino": int(no_treino.sum()),
        "n_safras_teste": int((~no_treino).sum()),
        "proporcao_treino_alvo": cfg["divisao"]["proporcao_treino"],
        "proporcao_treino_realizada": round(float(tabela.loc[no_treino, "n"].sum()
                                                  / tabela["n"].sum()), 4),
    }
    return base, resumo, tabela

## 6. Resumos da base

Primeira leitura da base pronta: volume, taxa (PD) ou média (LGD) e safras de cada partição, e a mesma conta safra a safra. Diferenças grandes de taxa entre treino, teste e OOT já indicam o que esperar do backtest.

Na PD aparecem duas taxas:

- `taxa_inadimplencia_12m`: média do `alvo_12m` (só safras maduras);
- `taxa_default_ate_corte`: fração de contratos que entraram em default **em qualquer idade** até a data de corte. Útil para o HIST, que é usado na PD lifetime.

In [ ]:
def resumo_por_particao(base, cfg):
    """Volume, taxa (PD) ou LGD média (LGD) e safras por partição."""
    colunas = cfg["colunas"]
    agregacoes = [
        F.count(F.lit(1)).alias("contratos"),
        F.countDistinct(colunas["safra"]).alias("n_safras"),
        F.min(colunas["safra"]).alias("primeira_safra"),
        F.max(colunas["safra"]).alias("ultima_safra"),
    ]
    if cfg["tarefa"] == "pd":
        agregacoes += [F.avg(colunas["alvo"]).alias("taxa_inadimplencia_12m"),
                       F.avg("evento_default").alias("taxa_default_ate_corte")]
    else:
        agregacoes += [F.avg(colunas["alvo"]).alias("lgd_media"),
                       F.stddev(colunas["alvo"]).alias("lgd_desvio_padrao")]
    tabela = base.groupBy("particao").agg(*agregacoes).toPandas()
    tabela["participacao_volume"] = tabela["contratos"] / tabela["contratos"].sum()
    tabela["ordem"] = tabela["particao"].map({p: i for i, p in enumerate(ORDEM_PARTICOES)})
    return tabela.sort_values("ordem").drop(columns="ordem").reset_index(drop=True)


def resumo_por_safra(base, cfg):
    """Volume e taxa (ou LGD média) safra a safra, com a partição de cada safra."""
    colunas = cfg["colunas"]
    if cfg["tarefa"] == "pd":
        agregacoes = [F.count(F.lit(1)).alias("contratos"),
                      F.avg(colunas["alvo"]).alias("taxa_inadimplencia_12m"),
                      F.avg("evento_default").alias("taxa_default_ate_corte")]
    else:
        agregacoes = [F.count(F.lit(1)).alias("contratos"),
                      F.avg(colunas["alvo"]).alias("lgd_media")]
    return (base.groupBy(colunas["safra"], "particao").agg(*agregacoes)
                .orderBy(colunas["safra"]).toPandas())

## 7. Execução da etapa

`executa_etapa_entrada` encadeia os passos acima, exibe os resultados e grava as tabelas Delta (`entrada_validacao`, `entrada_safras_excluidas`, `entrada_divisao`, `entrada_resumo_particao`, `entrada_resumo_safra`).

A base devolvida fica em cache (`persist`), porque todas as etapas seguintes a reutilizam.

In [ ]:
def executa_etapa_entrada(spark, sdf, cfg, gravar=True):
    """
    Etapa 1 completa.

    Parâmetros
        spark   sessão Spark (no Databricks, a variável global `spark`).
        sdf     base de entrada (Spark DataFrame) no contrato de dados.
        cfg     configuração criada por cria_configuracao.
        gravar  grava as tabelas de resultado em Delta (exige esquema_saida).

    Devolve
        Dicionário com:
          base                  Spark DataFrame com colunas derivadas e `particao`
          variaveis             lista de variáveis explicativas candidatas
          relatorio_validacao   pandas, uma linha por verificação
          safras_excluidas      pandas, contratos excluídos por safra imatura
          divisao               dicionário com o corte treino/teste
          tabela_divisao        pandas, volume e partição de cada safra do DES
          resumo_particao       pandas
          resumo_safra          pandas
    """
    variaveis = lista_variaveis_explicativas(sdf, cfg)
    print(f"[entrada] tarefa = {cfg['tarefa']} | {len(variaveis)} variáveis explicativas candidatas")

    relatorio = valida_contrato_entrada(sdf, cfg, variaveis)
    exibe(relatorio, "Verificações do contrato de dados")
    interrompe_se_falha(relatorio)

    base = adiciona_colunas_derivadas(sdf, cfg)
    base, safras_excluidas = separa_safras_imaturas(base, cfg)
    if len(safras_excluidas):
        exibe(safras_excluidas, "Contratos excluídos por safra imatura")

    base, divisao, tabela_divisao = divide_treino_teste_por_safra(base, cfg)
    base = base.persist()
    print(f"\n[entrada] treino até {divisao['ultima_safra_treino']} "
          f"({divisao['n_safras_treino']} safras, {divisao['proporcao_treino_realizada']:.1%} do DES) | "
          f"teste a partir de {divisao['primeira_safra_teste']} ({divisao['n_safras_teste']} safras)")

    resumo_particao = resumo_por_particao(base, cfg)
    resumo_safra = resumo_por_safra(base, cfg)
    exibe(resumo_particao, "Resumo por partição")
    exibe(resumo_safra, "Resumo por safra")

    if gravar:
        tabela_divisao_gravar = tabela_divisao.assign(**{k: str(v) for k, v in divisao.items()})
        for nome, tabela in [("entrada_validacao", relatorio),
                             ("entrada_safras_excluidas", safras_excluidas),
                             ("entrada_divisao", tabela_divisao_gravar),
                             ("entrada_resumo_particao", resumo_particao),
                             ("entrada_resumo_safra", resumo_safra)]:
            if len(tabela):
                grava_tabela_delta(spark, tabela, nome, cfg)

    return {
        "base": base,
        "variaveis": variaveis,
        "relatorio_validacao": relatorio,
        "safras_excluidas": safras_excluidas,
        "divisao": divisao,
        "tabela_divisao": tabela_divisao,
        "resumo_particao": resumo_particao,
        "resumo_safra": resumo_safra,
    }

## 8. Exemplo de uso

```python
cfg = cria_configuracao(tarefa="pd", metrica="ks", data_corte="2025-09-01",
                        esquema_saida="catalogo.risco_desafiante")
base_entrada = spark.table("catalogo.risco.base_pd_desafiante")
etapa1 = executa_etapa_entrada(spark, base_entrada, cfg)

base = etapa1["base"]            # segue para a etapa 2
variaveis = etapa1["variaveis"]
```